
# 06 — Model Evaluation

## Objective

Evaluate the trained CNN on unseen test images.

We calculate:

- Test accuracy
- Precision per class
- Recall per class
- F1-score per class
- Confusion matrix

Accuracy alone is not sufficient to understand a
medical-image classifier.

Recall is especially important to inspect because
it shows how frequently examples of a particular
class are correctly identified.

The dataset may contain acquisition-related biases,
and we have not verified patient-level independence.
Therefore, test results describe performance on this
dataset rather than clinical diagnostic reliability.


In [ ]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.brainscan import (
    create_dataset,
    CLASS_NAMES,
    CLASS_TO_ID
)

manifest = pd.read_csv(
    PROJECT_ROOT / "data/splits/split_manifest.csv"
)

test_rows = manifest[manifest["split"] == "test"]

test_records = [
    (PROJECT_ROOT / path, label)
    for path, label in zip(
        test_rows["path"],
        test_rows["label"]
    )
]

test_ds = create_dataset(
    test_records,
    training=False
)

model = tf.keras.models.load_model(
    PROJECT_ROOT / "models/best_cnn.keras"
)

print("Test images:", len(test_records))


In [ ]:

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")

probabilities = model.predict(test_ds, verbose=1)

y_pred = np.argmax(probabilities, axis=1)

y_true = np.array([
    CLASS_TO_ID[label]
    for path, label in test_records
])

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    zero_division=0
)

print("\nClassification Report:")
print(report)


In [ ]:

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES)))
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

display.plot(
    cmap="Blues",
    xticks_rotation=45
)

plt.title("BrainScan AI — Confusion Matrix")
plt.tight_layout()
plt.show()

report_path = PROJECT_ROOT / "models/test_report.txt"
report_path.write_text(
    f"Test accuracy: {test_accuracy:.4f}\n\n{report}"
)

print("Report saved:", report_path)
